<a href="https://colab.research.google.com/github/kkumarisfdc/Kiran-s_Portfolio/blob/main/Hyperparameter_tuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# -*- coding: utf-8 -*-

"""colab-"Hyperparameter tuning"-dr_bhupender-08_07_2026.ipynb

Automatically generated by Colab.

Hyperparameter Optimization for sgRNA Selection

1. Importing the Toolkits

"""

# --- STEP 1: IMPORTING PACKAGES ---

# Data manipulation libraries

import numpy as np

import pandas as pd

In [ ]:

# Machine learning model split and tuning tools

from sklearn.model_selection import train_test_split, GridSearchCV

# Ensemble classifier architecture

from sklearn.ensemble import RandomForestClassifier

# Robust evaluation metrics for biological data profiling

from sklearn.metrics import classification_report, matthews_corrcoef, make_scorer

print("All libraries successfully loaded!")


All libraries successfully loaded!


In [ ]:

"""2. Loading the CRISPR Dataset

GC_Percent: The nucleotide base composition (guides fail if binding stability is too low or too high).

Hairpin_Energy: Thermodynamic folding scores (stable internal loops prevent proper DNA targeting).

Target_Cleavage: The true lab result (1 = High-efficiency edit, 0 = Failed/inefficient edit).

"""

# --- STEP 2: DATASET ---

# Constructing a dataframe matrix based on experimental Cas9 editing runs

crispr_dataset = pd.DataFrame({

'GC_Percent': [0.45, 0.52, 0.31, 0.58, 0.65, 0.42, 0.50, 0.38, 0.48, 0.62, 0.55, 0.41, 0.35, 0.51, 0.60, 0.47, 0.53, 0.33, 0.49, 0.57, 0.44, 0.59, 0.39, 0.51, 0.63],

'Hairpin_Energy': [-2.1, -1.4, -0.5, -3.2, -4.1, -1.8, -1.1, -0.8, -2.5, -3.8, -1.9, -1.5, -0.6, -2.2, -3.5, -1.7, -1.2, -0.4, -2.0, -3.1, -1.6, -3.4, -0.9, -2.1, -3.9],

'Target_Cleavage': [1, 1, 0, 1, 0, 1, 1, 0, 1, 0, 1, 1, 0, 1, 0, 1, 1, 0, 1, 1, 1, 1, 0, 1, 0]

})

# Display basic structural dimensions of our experimental screen matrix

print(f"Data successfully compiled! Shape: {crispr_dataset.shape} (Rows, Columns)")

print("\nFirst 5 sgRNA screening profiles:")

crispr_dataset.head()


Data successfully compiled! Shape: (25, 3) (Rows, Columns)

First 5 sgRNA screening profiles:


,GC_Percent,Hairpin_Energy,Target_Cleavage
0,0.45,-2.1,1
1,0.52,-1.4,1
2,0.31,-0.5,0
3,0.58,-3.2,1
4,0.65,-4.1,0


In [ ]:
"""3. Dataset Features & Stratified Splitting

To train a model without bias, we separate the biological characteristics (our input matrix, $X$) from the true cleavage outcomes (our label vector, $y$).

Then, we split our data into a Training Set (70% of the samples) and a Test Set (30% of the samples).

We use stratification to ensure that both our training and testing subsets contain an identical proportion of successful vs. failed edits.

"""

# --- STEP 3: MATRIX PARTITIONING & TRAIN-TEST SPLITTING ---

# Step 3.1: Extract independent sequence features into matrix X

X = crispr_dataset[['GC_Percent', 'Hairpin_Energy']]

# Step 3.2: Extract the experimental label target into vector y

y = crispr_dataset['Target_Cleavage']

# Step 3.3: Perform the stratified partition split

X_train, X_test, y_train, y_test = train_test_split(

X,

y,

test_size=0.30, # Reserve exactly 30% of our real data for the final test evaluation

random_state=42, # Locks the random split mechanics to yield stable replication

stratify=y # Guarantees identical success/failure ratio across both subsets

)

print(f"Training features structural shape: {X_train.shape}")

print(f"Independent validation shape : {X_test.shape}")

Training features structural shape: (17, 2)
Independent validation shape : (8, 2)


In [ ]:
"""4. Hyperparameter Grid Search Matrix

A Random Forest cannot optimize its own structural architecture (such as tree count or branch depth) during regular training loops. These choices are called hyperparameters.

We will configure GridSearchCV to test multiple hyperparameter settings automatically.

Since biological screens are often highly imbalanced, standard accuracy can be misleading.

To fix this, we will explicitly command our grid engine to search for the settings that produce the highest Matthews Correlation Coefficient (MCC) score across 3 internal data validation folds.

"""

# --- STEP 4: TUNING ENGINE SETUP ---

# Step 4.1: Initialize our generic base Random Forest architecture

base_rf_model = RandomForestClassifier(random_state=42)

# Step 4.2: Build a dictionary map defining parameter spaces to exhaustively evaluate

hyperparameter_grid = {

'n_estimators': [10, 20, 50], # Number of decision trees inside the forest ensemble

'max_depth': [3, 5, None] # Hard vertical split depth cutoffs per tree

}

# Step 4.3: Turn the standard MCC calculation function into a Scikit-Learn scoring engine

biological_mcc_scorer = make_scorer(matthews_corrcoef)

# Step 4.4: Link everything into a centralized Grid Search Cross-Validation manager

tuning_pipeline = GridSearchCV(

estimator=base_rf_model,

param_grid=hyperparameter_grid,

cv=3, # Deploy 3-Fold cross-validation loops

scoring=biological_mcc_scorer # Optimize for the Matthews Correlation Coefficient

)

print("Grid search cross-validation pipeline successfully constructed!")

Grid search cross-validation pipeline successfully constructed!


In [ ]:

"""5. Executing the Grid Search Optimization Loop"""

# --- STEP 5: MODEL TRAINING & METRIC OPTIMIZATION ---

# Run the complete cross-validation matrix training process over the training split

tuning_pipeline.fit(X_train, y_train)

print("\n--- TUNING EXECUTION RESULTS ---")

# Extract and display the absolute best hyperparameter setup discovered by the grid engine

print(f"Winning Parameter Layout Found: {tuning_pipeline.best_params_}")

print(f"Top Mean Cross-Validated MCC Validation Score achieved: {tuning_pipeline.best_score_:.4f}")

"""6. Model Extraction & Final Test Set Validation"""

# --- STEP 6: PERFORMANCE BENCHMARK EVALUATION ---

# Step 6.1: Isolate the final fine-tuned model instance from the tuning wrapper

optimized_crispr_rf = tuning_pipeline.best_estimator_

# Step 6.2: Compute binary cleavage success predictions on the unseen test set features

final_predictions = optimized_crispr_rf.predict(X_test)

print("=== FINAL MODEL PERFORMANCE EVALUATION ===")

# Step 6.3: Print out a multi-metric classification report (Precision, Recall, F1)

print(classification_report(y_test, final_predictions, target_names=['Low_Cleavage', 'High_Cleavage']))

# Step 6.4: Print out the final out-of-sample Matthews Correlation Coefficient (MCC) score

test_set_mcc = matthews_corrcoef(y_test, final_predictions)

print(f"Final Out-Of-Sample Matthews Correlation Coefficient (MCC): {test_set_mcc:.4f}")


--- TUNING EXECUTION RESULTS ---
Winning Parameter Layout Found: {'max_depth': 3, 'n_estimators': 20}
Top Mean Cross-Validated MCC Validation Score achieved: 0.5690
=== FINAL MODEL PERFORMANCE EVALUATION ===
               precision    recall  f1-score   support

 Low_Cleavage       1.00      1.00      1.00         3
High_Cleavage       1.00      1.00      1.00         5

     accuracy                           1.00         8
    macro avg       1.00      1.00      1.00         8
 weighted avg       1.00      1.00      1.00         8

Final Out-Of-Sample Matthews Correlation Coefficient (MCC): 1.0000
